# Vector & Metric Spaces u AI: od teorije do prakse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boba1987/vector-spaces/blob/master/vector_metric_spaces_ai_colab.ipynb)

Ovaj notebook povezuje **dve matematicke teorije** u jedan AI pipeline (semantic retrieval + kNN klasifikacija):

- **Gilbert Strang, _Linear Algebra and Its Applications_, Chapter 2 – Vector Spaces**: gde "zive" embeddingi (vektorski prostor, baza, nullspace, cetiri fundamentalna podprostora, projekcija).
- **_Introduction to Analysis_, Chapter 7 – Metric Spaces**: kako merimo slicnost/rastojanje i zasto trening uopste konvergira (metrike, norme, nejednakosti, Cauchy nizovi, kompletnost).

## Vodeci princip

> **Definicija / Teorema iz knjige  ->  numericki dokaz da zaista vazi  ->  upotreba te iste teorije u prakticnom AI zadatku.**

Svaka sekcija ima **STA** radimo, **KAKO** (formula + kod) i **ZASTO** (referenca na konkretnu sekciju/definiciju/teoremu iz PDF-a). Notebook ispisuje detaljne logove (`[LOG]`, `[MATRIX]`, `[CHECK]`, `[PROOF]`) i crta grafike sa akcentom na matematiku.

## Mapa referenci (kod  ->  knjiga)

| Sekcija u notebooku | Knjiga | Referenca |
|---|---|---|
| 3.1 Aksiomi vektorskog prostora | Strang Ch2 | 2.1 Vector Spaces and Subspaces |
| 3.2 Nezavisnost, rang, baza (RREF) | Strang Ch2 | 2.3 Linear Independence, Basis, Dimension |
| 3.3 Resavanje Ax=0 i Ax=b | Strang Ch2 | 2.2 Solving Ax=0 and Ax=b |
| 3.4 Cetiri fundamentalna podprostora | Strang Ch2 | 2.4 The Four Fundamental Subspaces |
| 3.5 SVD, projekciona matrica, residual | Strang Ch2 | 2.1 / 2.3 / 2.4 |
| 4.1 Metrike i aksiomi | Analysis Ch7 | Def 7.1, Ex 7.3–7.8 |
| 4.2 Norme i indukovana metrika | Analysis Ch7 | Def 7.11, Prop 7.12 |
| 4.3 Inner product i Cauchy-Schwarz | Analysis Ch7 | Ex 7.15, Thm 7.54 |
| 4.4 Minkowski / nejednakost trougla | Analysis Ch7 | Cor 7.55 |
| 4.5 Ekvivalencija normi | Analysis Ch7 | str. 98 |
| 4.6 Jedinicne lopte | Analysis Ch7 | Fig 1, Ex 7.20 |
| 4.7 Otvorene lopte / okoline | Analysis Ch7 | Def 7.18 |
| 4.8 Nizovi, Cauchy, kompletnost | Analysis Ch7 | Def 7.31 / 7.38 / 7.39 |
| 5. Retrieval po metrikama | obe | Ch2 (V_k) + Ch7 (metrike) |
| 6. kNN klasifikacija | obe | Ch7 (metrike) + nejednakost trougla |

PDF fajlovi: `Gilbert_Strang_Linear_Algebra_and_Its_Applications.pdf` (Ch2 ~str. 87–130) i `intro_analysis_ch7.pdf` (Ch7).

## 1. Setup i helperi

**STA:** instaliramo biblioteke, fiksiramo seed radi reproduktivnosti i definisemo helper funkcije za citljiv log i numericke dokaze.

**KAKO:** uvodimo cetiri tipa loga:
- `log_step(name, detail)` – sta je korak uradio,
- `log_matrix(name, M)` – oblik i preview matrice,
- `log_check(name, ok, detail)` – PASS/FAIL provera nekog svojstva,
- `log_proof(name, lhs, rhs, kind)` – numericki dokaz jednakosti/nejednakosti uz prikaz vrednosti i margine.

**ZASTO:** cilj projekta je da svaku teoremu prvo *numericki potvrdimo* pa onda iskoristimo; zato nam treba uniforman, glasan log koji jasno kaze da li teorija "drzi vodu".

In [ ]:
# Colab setup: instalacija je bezbedna i kada se pokrece lokalno.
# sympy se koristi za egzaktni RREF (3.2); ima i numericki fallback ako nije dostupan.
!pip -q install numpy pandas matplotlib scikit-learn scipy sympy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
np.random.seed(SEED)

# Konzistentan stil grafika kroz ceo notebook.
plt.rcParams.update({
    'figure.figsize': (8, 4.5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 12,
    'font.size': 10,
})


def log_step(step_name, details=''):
    print(f'\n[LOG] {step_name}')
    if details:
        print(f'      {details}')


def log_matrix(name, M, preview_rows=3, preview_cols=6):
    M = np.asarray(M)
    print(f'[MATRIX] {name}: shape={M.shape}, dtype={M.dtype}')
    if M.ndim == 1:
        print(f'[MATRIX] {name} preview:', np.round(M[:preview_cols], 4))
    else:
        r = min(preview_rows, M.shape[0])
        c = min(preview_cols, M.shape[1])
        display(pd.DataFrame(np.round(M[:r, :c], 4)))


def log_check(name, ok, detail=''):
    # Glasna PASS/FAIL provera nekog matematickog svojstva.
    status = 'PASS' if bool(ok) else 'FAIL'
    print(f'[CHECK] {name}: {status}', f'| {detail}' if detail else '')
    return bool(ok)


def log_proof(name, lhs, rhs, kind='le', tol=1e-8):
    # kind='eq' -> dokazujemo lhs == rhs; kind='le' -> dokazujemo lhs <= rhs.
    lhs = float(lhs)
    rhs = float(rhs)
    if kind == 'eq':
        ok = abs(lhs - rhs) <= tol + tol * max(abs(lhs), abs(rhs))
        rel = f'|lhs-rhs|={abs(lhs - rhs):.3e}'
        sym = '=='
    else:
        ok = lhs <= rhs + tol
        rel = f'margin(rhs-lhs)={rhs - lhs:.6f}'
        sym = '<='
    status = 'PASS' if ok else 'FAIL'
    print(f'[PROOF] {name}: {lhs:.6f} {sym} {rhs:.6f}  ->  {status}  ({rel})')
    return ok


log_step('Setup OK', f'SEED={SEED}. Helperi log_step / log_matrix / log_check / log_proof spremni.')

## 2. Dataset: mala kolekcija dokumenata sa temama

**STA:** pravimo kolekciju kratkih tekstova iz vise tema (`ai`, `linear_algebra`, `metric_spaces`, `sports`, `health`, `finance`). Teme = labele koje kasnije sluze za evaluaciju retrieval-a i kNN-a.

**KAKO:** svaki tekst -> vektor (embedding). Slaganjem svih vektora dobijamo matricu `X` oblika `(n_dokumenata, d_features)`. Default backend je TF-IDF (lagan i stabilan u Colab-u); opciono se moze ukljuciti `sentence-transformers`.

**ZASTO:** vise razlicitih tema daje matricu `X` sa vise nezavisnih "pravaca", pa ce sve teoreme (rang, nullspace, podprostori, metrike) imati netrivijalan i vidljiv efekat.

In [ ]:
documents = [
    # ai
    'Transformers use self-attention to model long-range dependencies in text.',
    'Gradient descent minimizes a loss function by iterative parameter updates.',
    'PCA projects data onto principal components with maximum variance.',
    'Convolutional neural networks are effective for image classification tasks.',
    # linear_algebra
    'A vector space is closed under addition and scalar multiplication.',
    'Linear independence means no vector is a combination of the others.',
    'Basis vectors span a subspace and define its dimension.',
    'The column space and the nullspace are two of the four fundamental subspaces.',
    # metric_spaces
    'A metric assigns a nonnegative distance to every pair of points.',
    'The triangle inequality bounds the distance via an intermediate point.',
    'A Cauchy sequence has terms that get arbitrarily close to each other.',
    'A complete metric space contains the limit of every Cauchy sequence.',
    # sports
    'Soccer teams optimize passing networks to create scoring opportunities.',
    'Marathon runners pace themselves to manage energy over long distances.',
    'Basketball defense relies on spacing and quick rotations.',
    # health
    'A healthy diet includes vegetables, proteins, and balanced micronutrients.',
    'Strength training improves muscle mass and insulin sensitivity.',
    'Adequate sleep supports memory consolidation and immune function.',
    # finance
    'Stock market volatility can increase during macroeconomic uncertainty.',
    'Diversification spreads risk across uncorrelated asset classes.',
    'Compound interest grows an investment exponentially over time.',
]

labels = [
    'ai', 'ai', 'ai', 'ai',
    'linear_algebra', 'linear_algebra', 'linear_algebra', 'linear_algebra',
    'metric_spaces', 'metric_spaces', 'metric_spaces', 'metric_spaces',
    'sports', 'sports', 'sports',
    'health', 'health', 'health',
    'finance', 'finance', 'finance',
]

assert len(documents) == len(labels), 'Broj dokumenata i labela mora biti jednak.'

df_docs = pd.DataFrame({'doc_id': range(len(documents)), 'text': documents, 'label': labels})

log_step('Dataset pripremljen', f'Dokumenata={len(documents)} | Tema={df_docs.label.nunique()}')
display(df_docs.groupby('label').size().rename('broj_dokumenata').to_frame())
display(df_docs.head(8))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Primarni backend: TF-IDF (lagan, deterministican). Opcioni: sentence-transformers.
USE_SENTENCE_TRANSFORMERS = False

if USE_SENTENCE_TRANSFORMERS:
    !pip -q install sentence-transformers
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    X = st_model.encode(documents, convert_to_numpy=True).astype(float)
    feature_names = [f'emb_{i}' for i in range(X.shape[1])]
    log_step('Embedding backend', 'sentence-transformers (semanticki bogatije).')
else:
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(documents).toarray().astype(float)
    feature_names = vectorizer.get_feature_names_out().tolist()
    log_step('Embedding backend', 'TF-IDF (stabilno za Colab).')

y = np.array(labels)

# Train/test split koristimo kasnije za kNN (6). Stratifikujemo po label-i.
idx_all = np.arange(X.shape[0])
idx_train, idx_test = train_test_split(
    idx_all, test_size=0.30, random_state=SEED, stratify=y
)

log_step('Embedding matrica X kreirana', f'X.shape={X.shape} | rank(X)={np.linalg.matrix_rank(X)}')
log_step('Train/test split', f'train={len(idx_train)} | test={len(idx_test)}')
log_matrix('X', X)

# 3. Vector Spaces (Strang, Chapter 2)

U ovoj fazi tretiramo embeddinge kao elemente vektorskog prostora i, redom, numericki potvrdjujemo svojstva iz poglavlja 2 pre nego sto ih iskoristimo.

## 3.1 Aksiomi vektorskog prostora (2.1)

**STA:** proveravamo da redovi/vektori embeddinga zaista zadovoljavaju aksiome vektorskog prostora.

**KAKO:** za vektore `u, v` i skalare `a, b` proveravamo:
- zatvorenost: `u + v` i `a*u` su isto dimenzije (ostaju u prostoru),
- komutativnost i asocijativnost sabiranja,
- postojanje nule: `u + 0 = u`,
- postojanje inverza: `u + (-u) = 0`,
- distributivnost: `a*(u+v) = a*u + a*v` i `(a+b)*u = a*u + b*u`.

**ZASTO (Strang 2.1):** Strang definise vektorski prostor preko upravo ovih pravila zatvorenosti i aksioma. Tek kada znamo da embeddingi cine vektorski prostor, *smemo* slobodno da ih sabiramo, skaliramo i projektujemo - sto je temelj svega kasnije (PCA, projekcije, srednje vrednosti klasa).

In [ ]:
# Uzimamo dva embedding vektora i dva skalara, pa proveravamo aksiome (Strang 2.1).
u = X[0].copy()
v = X[1].copy()
a, b = 2.5, -1.3
zero = np.zeros_like(u)

log_step('3.1 Provera aksioma vektorskog prostora', f'dim(u)={u.shape[0]}, skalari a={a}, b={b}')

log_check('Zatvorenost na sabiranje (u+v ostaje u R^d)', (u + v).shape == u.shape)
log_check('Zatvorenost na skalarno mnozenje (a*u ostaje u R^d)', (a * u).shape == u.shape)
log_check('Komutativnost: u+v == v+u', np.allclose(u + v, v + u))
log_check('Asocijativnost: (u+v)+zero == u+(v+zero)', np.allclose((u + v) + zero, u + (v + zero)))
log_check('Neutral (nula): u+0 == u', np.allclose(u + zero, u))
log_check('Inverz: u+(-u) == 0', np.allclose(u + (-u), zero))
log_check('Distributivnost a*(u+v) == a*u+a*v', np.allclose(a * (u + v), a * u + a * v))
log_check('Distributivnost (a+b)*u == a*u+b*u', np.allclose((a + b) * u, a * u + b * u))
log_check('Asocijativnost skalara a*(b*u) == (a*b)*u', np.allclose(a * (b * u), (a * b) * u))
log_check('Jedinica: 1*u == u', np.allclose(1.0 * u, u))

print('\n=> Embedding vektori cine vektorski prostor R^d; smemo da ih sabiramo, skaliramo i projektujemo.')

## 3.2 Linearna nezavisnost, rang i baza preko RREF (2.3)

**STA:** trazimo koliko *nezavisnih* pravaca zaista postoji u nasim podacima i izdvajamo eksplicitnu **bazu prostora kolona**.

**KAKO:** radimo nad transponovanom matricom `A = X^T` (kolone = dokumenti). Reduced Row Echelon Form (RREF) otkriva **pivot kolone** - one cine bazu prostora kolona. Broj pivota = `rank`.

**ZASTO (Strang 2.3):** Strang definise bazu kao maksimalan skup linearno nezavisnih vektora koji razapinju prostor; broj baznih vektora = dimenzija. Pivot kolone iz eliminacije su upravo takav skup. U praksi nam ovo govori *stvarnu* dimenzionalnost reprezentacije (npr. koliko tema/koncepata je linearno razdvojivo), sto je granica kapaciteta modela.

In [ ]:
def numeric_rref(M, tol=1e-9):
    # Gauss-Jordan eliminacija sa parcijalnim pivotiranjem.
    # Vraca (RREF matricu, listu pivot kolona). Deterministicno i brzo za bilo koju velicinu.
    A = M.astype(float).copy()
    rows, cols = A.shape
    pivots = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        # Najveci element u koloni c ispod trenutnog reda -> stabilan pivot.
        pivot_row = r + np.argmax(np.abs(A[r:, c]))
        if abs(A[pivot_row, c]) <= tol:
            continue
        A[[r, pivot_row]] = A[[pivot_row, r]]
        A[r] = A[r] / A[r, c]
        for rr in range(rows):
            if rr != r:
                A[rr] = A[rr] - A[rr, c] * A[r]
        pivots.append(c)
        r += 1
    return A, pivots


# A = X: kolone su feature-pravci u R^n; trazimo bazu prostora kolona C(A) (Strang 2.3).
A = X
rref_A, pivot_cols = numeric_rref(A)

rank_rref = len(pivot_cols)
rank_numpy = int(np.linalg.matrix_rank(A))

# Baza prostora kolona = ORIGINALNE pivot kolone (ne RREF kolone).
col_space_basis = A[:, pivot_cols]

log_step('3.2 RREF / baza prostora kolona', f'X.shape={X.shape}')
log_check('rank(RREF) == rank(numpy)', rank_rref == rank_numpy,
          f'rank={rank_rref} (od max {min(A.shape)} mogucih pravaca)')
print(f'[INFO] Broj pivot kolona (nezavisnih feature-pravaca): {rank_rref}')
print(f'[INFO] Prvih nekoliko pivot indeksa: {pivot_cols[:10]}')
log_matrix('Baza prostora kolona (pivot kolone iz X)', col_space_basis)

# Provera: nepivot kolona je linearna kombinacija baze (tj. zaista zavisna).
if rank_rref < A.shape[1]:
    dep_idx = [c for c in range(A.shape[1]) if c not in pivot_cols][0]
    coeffs, *_ = np.linalg.lstsq(col_space_basis, A[:, dep_idx], rcond=None)
    recon = col_space_basis @ coeffs
    log_check(f'Nepivot kolona {dep_idx} = lin. kombinacija baze',
              np.allclose(recon, A[:, dep_idx], atol=1e-6),
              f'rezidual={np.linalg.norm(recon - A[:, dep_idx]):.2e}')
print('\n=> rank = stvarni broj nezavisnih pravaca = efektivna dimenzionalnost reprezentacije.')

## 3.3 Resavanje Ax=0 i Ax=b (2.2)

**STA:** racunamo **nullspace** `N(X) = {x : Xx = 0}` (homogeno resenje) i resavamo nehomogeni sistem `Xx = b` (least squares).

**KAKO:**
- `scipy.linalg.null_space(X)` daje ortonormiranu bazu nullspace-a; proveravamo `X @ n ≈ 0` za svaki bazni vektor.
- za `Xx = b` koristimo `np.linalg.lstsq` (najbolja aproksimacija kada egzaktno resenje ne postoji).

**ZASTO (Strang 2.2):** Strang razdvaja resenje sistema na *partikularno* (`Xx=b`) plus *homogeno* (`Xx=0`, ceo nullspace). Nullspace su pravci na koje je preslikavanje "slepo" - dodavanje bilo kog elementa nullspace-a ne menja izlaz. U AI ovo objasnjava redundansu/nejedinstvenost: razlicite kombinacije feature-a daju isti rezultat.

In [ ]:
from scipy.linalg import null_space

# --- Ax=0: baza nullspace-a N(X) ---
N = null_space(X)  # oblik (d, d-rank): kolone su ortonormirana baza nullspace-a
log_step('3.3 Ax=0 (nullspace)', f'dim N(X) = {N.shape[1]}  (= d - rank = {X.shape[1]} - {np.linalg.matrix_rank(X)})')

if N.shape[1] > 0:
    residuals = np.linalg.norm(X @ N, axis=0)  # ||X n_j|| za svaki bazni vektor
    log_check('Svi bazni vektori nullspace-a daju X@n ~ 0',
              np.allclose(X @ N, 0, atol=1e-8),
              f'max ||X@n||={residuals.max():.2e}')
    # Dodavanje elementa nullspace-a ne menja izlaz X@x.
    x_rand = np.random.randn(X.shape[1])
    n_vec = N[:, 0]
    log_proof('Invarijantnost: ||X(x+n) - Xx||', np.linalg.norm(X @ (x_rand + n_vec) - X @ x_rand), 0.0, kind='eq')
else:
    print('[INFO] Nullspace je trivijalan (samo 0) - kolone su nezavisne.')

# --- Ax=b: nehomogeni sistem preko least squares ---
b = X @ np.ones(X.shape[1])  # konstruisemo b koji je sigurno u prostoru kolona
x_ls, *_ = np.linalg.lstsq(X, b, rcond=None)
ls_residual = np.linalg.norm(X @ x_ls - b)
log_step('Ax=b (least squares)', f'||X x* - b|| = {ls_residual:.2e}')
log_check('Least-squares resenje rekonstruise b (b je u C(X))', ls_residual < 1e-6)

# Partikularno + homogeno = i dalje resenje (Strang 2.2): x* + n resava isti sistem.
if N.shape[1] > 0:
    x_alt = x_ls + 3.7 * N[:, 0]
    log_proof('(x* + n) je takodje resenje: ||X(x*+n) - b||', np.linalg.norm(X @ x_alt - b), ls_residual, kind='eq', tol=1e-6)
print('\n=> Resenje = partikularno (Xx=b) + ceo nullspace (Xx=0); nullspace = "slepi" pravci.')

## 3.4 Cetiri fundamentalna podprostora (2.4)

**STA:** za matricu `X` (oblik `m x n`, `m` dokumenata, `n` feature-a) racunamo dimenzije i ortogonalne odnose cetiri podprostora:
- prostor kolona `C(X) ⊂ R^m` (dim = r),
- nullspace `N(X) ⊂ R^n` (dim = n − r),
- prostor redova `C(X^T) ⊂ R^n` (dim = r),
- levi nullspace `N(X^T) ⊂ R^m` (dim = m − r).

**KAKO:** baze racunamo iz SVD/`null_space`. Proveravamo **rank-nullity**: `r + (n − r) = n`, i **ortogonalnost**: `N(X) ⊥ C(X^T)` i `N(X^T) ⊥ C(X)` (skalarni proizvodi ≈ 0).

**ZASTO (Strang 2.4):** ovo je "Veliki Slika" teoreme linearne algebre - cela teorija sistema staje u ova cetiri podprostora i dva para ortogonalnih komplemenata. U AI to znaci: prostor redova nosi koristan signal, nullspace je redundansa; razumevanje ove dekompozicije je osnova za PCA/projekcije koje radimo u 3.5.

In [ ]:
from scipy.linalg import null_space, orth

m, n = X.shape
r = int(np.linalg.matrix_rank(X))

# Baze cetiri fundamentalna podprostora.
col_space = orth(X)              # C(X)  ⊂ R^m,  dim r
row_space = orth(X.T)            # C(X^T) ⊂ R^n, dim r
null_X = null_space(X)           # N(X)   ⊂ R^n, dim n-r
left_null = null_space(X.T)      # N(X^T) ⊂ R^m, dim m-r

dims = {
    'C(X)  col space  (R^m)': col_space.shape[1],
    'N(X)  null space (R^n)': null_X.shape[1],
    'C(X^T) row space (R^n)': row_space.shape[1],
    'N(X^T) left null (R^m)': left_null.shape[1],
}

log_step('3.4 Cetiri fundamentalna podprostora', f'X: m={m} dokumenata, n={n} feature-a, rank r={r}')
for name, d in dims.items():
    print(f'   dim {name} = {d}')

# Rank-nullity teorema: r + (n - r) = n  i  r + (m - r) = m
log_proof('Rank-nullity (domen R^n): r + dim N(X)', r + null_X.shape[1], n, kind='eq')
log_proof('Rank-nullity (kodomen R^m): r + dim N(X^T)', r + left_null.shape[1], m, kind='eq')

# Ortogonalnost parova komplemenata (Strang 2.4).
def max_abs_inner(B1, B2):
    if B1.shape[1] == 0 or B2.shape[1] == 0:
        return 0.0
    return float(np.max(np.abs(B1.T @ B2)))

log_check('N(X) ⊥ C(X^T)  (red i nullspace ortogonalni u R^n)',
          max_abs_inner(null_X, row_space) < 1e-8,
          f'max|<.,.>|={max_abs_inner(null_X, row_space):.2e}')
log_check('N(X^T) ⊥ C(X)  (kolona i levi nullspace ortogonalni u R^m)',
          max_abs_inner(left_null, col_space) < 1e-8,
          f'max|<.,.>|={max_abs_inner(left_null, col_space):.2e}')

# Grafik dimenzija.
fig, ax = plt.subplots(figsize=(8, 4))
names = ['C(X)\n(R^m)', 'N(X)\n(R^n)', 'C(X^T)\n(R^n)', 'N(X^T)\n(R^m)']
vals = [col_space.shape[1], null_X.shape[1], row_space.shape[1], left_null.shape[1]]
ax.bar(names, vals)
for i, v in enumerate(vals):
    ax.text(i, v + 0.3, str(v), ha='center')
ax.set_ylabel('dimenzija')
ax.set_title('Dimenzije cetiri fundamentalna podprostora (Strang 2.4)')
plt.tight_layout(); plt.show()
print('=> Dva para ortogonalnih komplemenata; koristan signal je u prostoru redova, redundansa u nullspace-u.')

## 3.5 SVD, baza podprostora, projekciona matrica i residual (2.1 / 2.3 / 2.4)

**STA:** preko SVD `X = U S V^T` biramo top-`k` desnih singularnih vektora kao **ortonormiranu bazu** `V_k` dominantnog podprostora, gradimo **projekcionu matricu** `P = V_k V_k^T` i razlazemo svaki dokument na `x = P x + (I−P) x` (projekcija + residual).

**KAKO:** dokazujemo dve definicione osobine projekcije:
- **idempotentnost** `P^2 = P` (projektovati dvaput = jednom),
- **simetrija** `P = P^T` (ortogonalna projekcija).

Crtamo spektar singularnih vrednosti, "captured energy" u funkciji `k`, i 2D PCA scatter dokumenata.

**ZASTO (Strang 2.1/2.3/2.4):** `V_k` je konkretna baza podprostora (2.3); projekcija na prostor kolona i ostatak u ortogonalnom komplementu je upravo dekompozicija iz 2.4. Praksa: ovo je PCA/LSA - zadrzavamo dominantni signal, a residual koristimo kao meru "novine" i kasnije za subspace-aware retrieval (sekcija 5).

In [ ]:
# SVD: X = U S V^T. Desni singularni vektori (kolone V) -> ortonormirana baza feature prostora.
U, S, Vt = np.linalg.svd(X, full_matrices=False)

k = min(5, X.shape[1])
V_k = Vt[:k].T                 # (d, k) baza dominantnog podprostora
P = V_k @ V_k.T                # projekciona matrica (d, d)

captured_energy = np.sum(S[:k] ** 2) / np.sum(S ** 2)
log_step('3.5 SVD podprostor', f'k={k} | captured_energy={captured_energy:.4f} | V_k.shape={V_k.shape}')

# Definicione osobine ortogonalne projekcije.
log_proof('Idempotentnost: max|P^2 - P|', np.max(np.abs(P @ P - P)), 0.0, kind='eq')
log_proof('Simetrija: max|P - P^T|', np.max(np.abs(P - P.T)), 0.0, kind='eq')
log_check('Ortonormirana baza: V_k^T V_k = I', np.allclose(V_k.T @ V_k, np.eye(k), atol=1e-8))

# Projekcija + residual za sve dokumente.
X_proj = X @ P
X_res = X - X_proj
res_norms = np.linalg.norm(X_res, axis=1)

# Pitagora po podprostorima (2.4): ||x||^2 = ||Px||^2 + ||(I-P)x||^2.
i0 = 0
log_proof('Pitagora: ||x||^2 == ||Px||^2 + ||(I-P)x||^2',
          np.sum(X[i0] ** 2),
          np.sum(X_proj[i0] ** 2) + np.sum(X_res[i0] ** 2), kind='eq', tol=1e-6)

log_matrix('V_k (baza podprostora)', V_k)
display(pd.DataFrame({'doc_id': df_docs.doc_id, 'label': df_docs.label,
                      'residual_norm': np.round(res_norms, 4)}).head(8))

# --- Grafici ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(np.arange(1, len(S) + 1), S, marker='o')
axes[0].axvline(k, color='red', ls='--', alpha=0.6, label=f'k={k}')
axes[0].set_title('Spektar singularnih vrednosti (SVD)')
axes[0].set_xlabel('indeks'); axes[0].set_ylabel('singularna vrednost'); axes[0].legend()

energies = [np.sum(S[:kk] ** 2) / np.sum(S ** 2) for kk in range(1, len(S) + 1)]
axes[1].plot(np.arange(1, len(S) + 1), energies, marker='s')
axes[1].axhline(captured_energy, color='red', ls='--', alpha=0.6, label=f'k={k}: {captured_energy:.2f}')
axes[1].set_title('Captured energy vs dimenzija podprostora')
axes[1].set_xlabel('k'); axes[1].set_ylabel('udeo energije'); axes[1].set_ylim(0, 1.02); axes[1].legend()
plt.tight_layout(); plt.show()

# 2D PCA scatter dokumenata (projekcija na prva 2 singularna pravca).
coords2d = X @ Vt[:2].T
fig, ax = plt.subplots(figsize=(8, 6))
for lab in sorted(df_docs.label.unique()):
    mask = (y == lab)
    ax.scatter(coords2d[mask, 0], coords2d[mask, 1], label=lab, s=80)
for i in range(X.shape[0]):
    ax.annotate(str(i), (coords2d[i, 0], coords2d[i, 1]), fontsize=8, alpha=0.7)
ax.set_title('Dokumenti u 2D podprostoru (prva 2 singularna pravca)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# 4. Metric Spaces (Introduction to Analysis, Chapter 7)

Sada definisemo *kako merimo rastojanje/slicnost* izmedju embeddinga. Svaku meru prvo proveravamo da li je validna metrika, pa je koristimo.

## 4.1 Metrike i aksiomi (Def 7.1)

**STA:** implementiramo nekoliko metrika i numericki proveravamo da zadovoljavaju 3 aksioma iz Def 7.1.

**KAKO (Def 7.1):** funkcija `d : X × X → R` je metrika ako za sve `x, y, z`:
1. `d(x,y) ≥ 0` i `d(x,y)=0 ⇔ x=y`,
2. `d(x,y)=d(y,x)` (simetrija),
3. `d(x,y) ≤ d(x,z)+d(z,y)` (nejednakost trougla).

Implementiramo: `d_L1` (taxicab, Ex 7.6), `d_L2` (Euklidska, Ex 7.4/7.5), `d_Linf` (maksimum, Ex 7.7), `d_Lp` (Ex 7.15) i `d_discrete` (Ex 7.2). Proveravamo aksiome na mnogo slucajnih trojki tacaka.

**ZASTO:** retrieval i kNN se *u potpunosti* oslanjaju na meru rastojanja. Ako mera nije validna metrika (npr. krsi trougao), algoritmi pretrage/pruninga mogu da daju besmislene rezultate. Zato prvo dokazujemo da su mere ispravne.

In [ ]:
# Metrike iz Chapter 7.
def d_L1(x, y):      # Ex 7.6  (taxicab / l1)
    return np.sum(np.abs(x - y))

def d_L2(x, y):      # Ex 7.4 / 7.5  (Euklidska / l2)
    return np.sqrt(np.sum((x - y) ** 2))

def d_Linf(x, y):    # Ex 7.7  (maksimum / l_inf)
    return np.max(np.abs(x - y))

def d_Lp(x, y, p=3): # Ex 7.15  (l_p norma)
    return np.sum(np.abs(x - y) ** p) ** (1.0 / p)

def d_discrete(x, y, tol=1e-12):  # Ex 7.2  (diskretna metrika)
    return 0.0 if np.allclose(x, y, atol=tol) else 1.0

METRICS = {
    'L1 (taxicab)': d_L1,
    'L2 (euclidean)': d_L2,
    'Linf (max)': d_Linf,
    'L3 (Lp, p=3)': lambda x, y: d_Lp(x, y, p=3),
    'discrete': d_discrete,
}

def verify_metric_axioms(d, dim=6, trials=400, rng=None):
    rng = rng or np.random.default_rng(SEED)
    nonneg = identity = symmetry = triangle = True
    for _ in range(trials):
        x, y, z = rng.normal(size=dim), rng.normal(size=dim), rng.normal(size=dim)
        if d(x, y) < -1e-12:
            nonneg = False
        if abs(d(x, x)) > 1e-9 or (d(x, y) <= 1e-12 and not np.allclose(x, y)):
            identity = False
        if abs(d(x, y) - d(y, x)) > 1e-9:
            symmetry = False
        if d(x, y) > d(x, z) + d(z, y) + 1e-9:
            triangle = False
    return nonneg, identity, symmetry, triangle

log_step('4.1 Verifikacija aksioma metrike (Def 7.1)', f'{len(METRICS)} mera, po 400 slucajnih trojki')
rows = []
for name, d in METRICS.items():
    nn, idn, sym, tri = verify_metric_axioms(d)
    all_ok = nn and idn and sym and tri
    log_check(f'{name}: aksiom1(nonneg+id), aksiom2(sym), aksiom3(triangle)',
              all_ok, f'nonneg={nn}, id={idn}, sym={sym}, triangle={tri}')
    rows.append({'metric': name, 'nonneg+id': nn, 'symmetry': sym, 'triangle': tri, 'valid_metric': all_ok})

display(pd.DataFrame(rows))
print('=> Sve nabrojane mere su validne metrike -> mozemo ih bezbedno koristiti u retrieval-u i kNN-u.')

## 4.2 Norme i indukovana metrika (Def 7.11, Prop 7.12)

**STA:** uvodimo norme `||·||_1, ||·||_2, ||·||_inf` i pokazujemo da svaka norma indukuje metriku `d(x,y) = ||x − y||`, sa dodatnim svojstvima translacione invarijantnosti i homogenosti.

**KAKO (Def 7.11):** norma zadovoljava: (1) `||x|| ≥ 0`, `=0 ⇔ x=0`; (2) `||k x|| = |k| · ||x||`; (3) `||x+y|| ≤ ||x|| + ||y||`.
- **Prop 7.12:** `d(x,y) = ||x−y||` je metrika.
- str. 96: metrika iz norme je *translaciono invarijantna* `d(x+z, y+z) = d(x,y)` i *homogena* `d(kx, ky) = |k| d(x,y)`.

**ZASTO:** ova svojstva opravdavaju standardne preprocessing korake u ML-u: **centriranje** (oduzimanje srednje vrednosti) ne menja rastojanja (translaciona invarijantnost), a **skaliranje** ih menja predvidljivo (homogenost). Zato znamo tacno sta normalizacija radi metrici.

In [ ]:
def norm_p(x, p=2):
    if p == np.inf:
        return np.max(np.abs(x))
    return np.sum(np.abs(x) ** p) ** (1.0 / p)

rng = np.random.default_rng(SEED)
x = rng.normal(size=8)
yv = rng.normal(size=8)
zv = rng.normal(size=8)
ksc = -2.4

log_step('4.2 Norme i indukovana metrika', 'provera Def 7.11, Prop 7.12 i svojstava sa str. 96')

# Aksiomi norme (na primeru L2).
log_proof('Norma homogenost: ||k x|| == |k| ||x||', norm_p(ksc * x, 2), abs(ksc) * norm_p(x, 2), kind='eq')
log_proof('Norma trougao: ||x+y|| <= ||x||+||y||', norm_p(x + yv, 2), norm_p(x, 2) + norm_p(yv, 2), kind='le')
log_check('Pozitivna definitnost: ||x||=0 <=> x=0',
          abs(norm_p(np.zeros(8), 2)) < 1e-12 and norm_p(x, 2) > 0)

# Prop 7.12: d(x,y) = ||x - y|| je metrika -> proverimo trougao preko norme.
d_from_norm = lambda a, b: norm_p(a - b, 2)
log_proof('Indukovana metrika (trougao): d(x,y) <= d(x,z)+d(z,y)',
          d_from_norm(x, yv), d_from_norm(x, zv) + d_from_norm(zv, yv), kind='le')

# str. 96: translaciona invarijantnost i homogenost metrike.
log_proof('Translaciona invarijantnost: d(x+z, y+z) == d(x,y)',
          d_from_norm(x + zv, yv + zv), d_from_norm(x, yv), kind='eq')
log_proof('Homogenost metrike: d(kx, ky) == |k| d(x,y)',
          d_from_norm(ksc * x, ksc * yv), abs(ksc) * d_from_norm(x, yv), kind='eq')

print('\n=> Centriranje (translacija) NE menja rastojanja; skaliranje ih mnozi sa |k|. Zato znamo sta normalizacija radi.')

## 4.3 Inner product i Cauchy-Schwarz (Ex 7.15, Thm 7.54)

**STA:** definisemo skalarni proizvod `<x,y> = Σ x_i y_i`, dokazujemo Cauchy-Schwarz nejednakost i iz nje izvodimo **cosine slicnost**.

**KAKO:**
- Inner product i norma (Ex 7.15): `||x||_2 = sqrt(<x,x>)`.
- **Cauchy-Schwarz (Thm 7.54):** `|<x,y>| ≤ ||x||·||y||`. Proveravamo na mnogo parova i merimo marginu.
- Cosine: `cos(x,y) = <x,y> / (||x||·||y||)`. Iz Cauchy-Schwarz sledi da je u `[-1, 1]`.

**ZASTO:** cosine slicnost je *najcesca* mera u semantic search-u i NLP-u. Cauchy-Schwarz je tacno razlog zasto je ona ogranicena na `[-1,1]` i interpretabilna kao kosinus ugla. Bez ove teoreme cosine ne bi imao smisla kao normalizovana slicnost.

In [ ]:
def inner(x, y):
    return float(np.dot(x, y))

def cosine_sim(x, y, eps=1e-12):
    return inner(x, y) / (norm_p(x, 2) * norm_p(y, 2) + eps)

log_step('4.3 Inner product i Cauchy-Schwarz (Thm 7.54)', 'provera |<x,y>| <= ||x|| ||y||')

# Inner product indukuje L2 normu (Ex 7.15): ||x|| = sqrt(<x,x>).
xv = rng.normal(size=10)
log_proof('Norma iz inner product-a: ||x|| == sqrt(<x,x>)', norm_p(xv, 2), np.sqrt(inner(xv, xv)), kind='eq')

# Cauchy-Schwarz na mnogo slucajnih parova + minimalna margina.
margins, cosines = [], []
all_cs_ok = True
for _ in range(500):
    a, b = rng.normal(size=10), rng.normal(size=10)
    lhs = abs(inner(a, b))
    rhs = norm_p(a, 2) * norm_p(b, 2)
    margins.append(rhs - lhs)
    cosines.append(cosine_sim(a, b))
    if lhs > rhs + 1e-9:
        all_cs_ok = False

log_check('Cauchy-Schwarz vazi za svih 500 parova', all_cs_ok, f'min margina (rhs-lhs)={min(margins):.3e}')
log_check('Cosine slicnost je u [-1, 1] (posledica Cauchy-Schwarz)',
          (min(cosines) >= -1 - 1e-9) and (max(cosines) <= 1 + 1e-9),
          f'opseg=[{min(cosines):.4f}, {max(cosines):.4f}]')

# Vizualizacija: |<x,y>| naspram ||x|| ||y|| - sve tacke su ispod (ili na) dijagonali.
rng2 = np.random.default_rng(SEED)
pairs = [(rng2.normal(size=10), rng2.normal(size=10)) for _ in range(300)]
lhs_vals = np.array([abs(inner(a, b)) for a, b in pairs])
rhs_vals = np.array([norm_p(a, 2) * norm_p(b, 2) for a, b in pairs])

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(rhs_vals, lhs_vals, s=18, alpha=0.6)
lim = max(rhs_vals.max(), lhs_vals.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', label='granica |<x,y>| = ||x|| ||y||')
ax.set_xlabel('||x|| · ||y||'); ax.set_ylabel('|<x, y>|')
ax.set_title('Cauchy-Schwarz: sve tacke ispod dijagonale (Thm 7.54)')
ax.legend(); plt.tight_layout(); plt.show()
print('=> Cosine slicnost = <x,y>/(||x|| ||y||) je zato uvek u [-1,1] i meri kosinus ugla.')

## 4.4 Minkowski / nejednakost trougla (Cor 7.55)

**STA:** proveravamo Minkowski nejednakost `||x+y||_p ≤ ||x||_p + ||y||_p` (nejednakost trougla za norme) za razne `p`, i merimo marginu.

**KAKO (Cor 7.55):** za `p=2` Minkowski sledi direktno iz Cauchy-Schwarz (dokaz u Appendix-u Ch7). Proveravamo i opste `p ∈ {1, 2, 3, ∞}` na slucajnim vektorima.

**ZASTO:** nejednakost trougla je *treci aksiom metrike* i kljucna je za korektnost kNN-a i tehnika ubrzanja pretrage (triangle-inequality pruning, npr. u ball-tree / metric indexing): ona garantuje da rastojanja "ne mogu da prevare", pa mozemo da odbacimo kandidate bez racunanja svih rastojanja.

In [ ]:
log_step('4.4 Minkowski / nejednakost trougla (Cor 7.55)', 'provera ||x+y||_p <= ||x||_p + ||y||_p')

p_values = [1, 2, 3, np.inf]
min_margins = {}
for p in p_values:
    margins = []
    ok = True
    for _ in range(500):
        a, b = rng.normal(size=12), rng.normal(size=12)
        lhs = norm_p(a + b, p)
        rhs = norm_p(a, p) + norm_p(b, p)
        margins.append(rhs - lhs)
        if lhs > rhs + 1e-9:
            ok = False
    min_margins[str(p)] = min(margins)
    log_check(f'Minkowski za p={p}', ok, f'min margina={min(margins):.3e}')

# Eksplicitan primer za p=2 (veza sa Cauchy-Schwarz iz Cor 7.55).
a, b = rng.normal(size=12), rng.normal(size=12)
log_proof('Primer p=2: ||x+y|| <= ||x|| + ||y||', norm_p(a + b, 2), norm_p(a, 2) + norm_p(b, 2), kind='le')

# Vizualizacija minimalne margine po p (uvek >= 0 -> nejednakost vazi).
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(p) for p in p_values], [min_margins[str(p)] for p in p_values])
ax.axhline(0, color='red', ls='--', alpha=0.6)
ax.set_xlabel('p (norma)'); ax.set_ylabel('min margina (rhs - lhs)')
ax.set_title('Minkowski: minimalna margina >= 0 za sve p')
plt.tight_layout(); plt.show()
print('=> Nejednakost trougla vazi -> kNN i triangle-inequality pruning su matematicki opravdani.')

## 4.5 Ekvivalencija normi (str. 98)

**STA:** proveravamo lanac nejednakosti `||x||_inf ≤ ||x||_2 ≤ ||x||_1 ≤ n·||x||_inf` i empirijski nalazimo konstante koje povezuju norme.

**KAKO (str. 98):** knjiga navodi `||x||_inf ≤ ||x||_2 ≤ ||x||_1 ≤ n·||x||_inf`. Norme su *ekvivalentne*: postoje konstante `c, C` takve da `c·||x||_a ≤ ||x||_b ≤ C·||x||_a`. Iz ovog sledi da definisu istu topologiju (iste otvorene skupove, granice, neprekidne funkcije).

**ZASTO:** zato izbor norme (L1 vs L2 vs Linf) menja *skalu* rastojanja, ali ne i *koja tacka konvergira* niti koje su tacke "blizu". U praksi: konvergencija treninga i pojam suseda su robusni na izbor norme - menja se konstanta, ne sustina.

In [ ]:
n_dim = 12
log_step('4.5 Ekvivalencija normi (str. 98)', f'provera ||x||inf <= ||x||2 <= ||x||1 <= n*||x||inf, n={n_dim}')

chain_ok = True
ratios_21, ratios_1inf = [], []  # za empirijske konstante
for _ in range(1000):
    v = rng.normal(size=n_dim)
    inf_, two_, one_ = norm_p(v, np.inf), norm_p(v, 2), norm_p(v, 1)
    if not (inf_ <= two_ + 1e-9 <= one_ + 1e-9 and one_ <= n_dim * inf_ + 1e-9):
        if not (inf_ <= two_ + 1e-9 and two_ <= one_ + 1e-9 and one_ <= n_dim * inf_ + 1e-9):
            chain_ok = False
    ratios_21.append(two_ / one_)
    ratios_1inf.append(one_ / inf_)

log_check('Lanac ||x||inf <= ||x||2 <= ||x||1 <= n||x||inf vazi (1000 vektora)', chain_ok)
print(f'[INFO] Empirijski: ||x||2/||x||1 ∈ [{min(ratios_21):.3f}, {max(ratios_21):.3f}]  (teorijski u [1/sqrt(n), 1])')
print(f'[INFO] Empirijski: ||x||1/||x||inf ∈ [{min(ratios_1inf):.3f}, {max(ratios_1inf):.3f}]  (teorijski u [1, n])')

# Vizualizacija: za 500 vektora, ||.||inf <= ||.||2 <= ||.||1 (sortirano radi citljivosti).
samp = np.array([[norm_p(v, np.inf), norm_p(v, 2), norm_p(v, 1)]
                 for v in rng.normal(size=(500, n_dim))])
order = np.argsort(samp[:, 1])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(samp[order, 0], label='||x||inf', alpha=0.8)
ax.plot(samp[order, 1], label='||x||2', alpha=0.8)
ax.plot(samp[order, 2], label='||x||1', alpha=0.8)
ax.set_title('Ekvivalencija normi: inf <= 2 <= 1 (sortirano po ||x||2)')
ax.set_xlabel('uzorak (sortiran)'); ax.set_ylabel('vrednost norme'); ax.legend()
plt.tight_layout(); plt.show()
print('=> Izbor norme menja skalu (konstantu), ali ne i topologiju: isti susedi, ista konvergencija.')

## 4.6 Jedinicne lopte u R^2 (Fig 1, Ex 7.20)

**STA:** crtamo granice jedinicnih lopti `B_1(0) = {x : ||x|| ≤ 1}` za L1, L2 i Linf norme.

**KAKO (Fig 1, Ex 7.20):** knjiga (Figure 1) prikazuje romb (L1), krug (L2) i kvadrat (Linf). Reprodukujemo to crtanjem skupa tacaka sa normom = 1.

**ZASTO:** oblik jedinicne lopte je *geometrijska intuicija* za to kako svaka metrika kaznjava odstupanja: L1 (romb) favorizuje retke (sparse) razlike, Linf (kvadrat) gleda samo najvecu komponentu, L2 (krug) je izotropan. Ovo direktno objasnjava ponasanje L1/L2/Linf u retrieval-u i kNN-u koji slede.

In [ ]:
log_step('4.6 Jedinicne lopte B1(0) u R^2 (Fig 1)', 'L1 (romb), L2 (krug), Linf (kvadrat)')

theta = np.linspace(0, 2 * np.pi, 720)
dirs = np.stack([np.cos(theta), np.sin(theta)], axis=1)  # pravci

def unit_ball_boundary(p):
    # Skaliramo svaki pravac tako da mu norma bude tacno 1.
    norms = np.array([norm_p(d, p) for d in dirs])
    return dirs / norms[:, None]

fig, ax = plt.subplots(figsize=(6.5, 6.5))
for p, name in [(1, 'L1 (romb)'), (2, 'L2 (krug)'), (np.inf, 'Linf (kvadrat)')]:
    b = unit_ball_boundary(p)
    ax.plot(b[:, 0], b[:, 1], label=name, linewidth=2)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_title('Granice jedinicnih lopti B1(0) (Ch7, Fig 1)')
ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.legend()
plt.tight_layout(); plt.show()

# Provera ugnjezdenja: Linf-lopta sadrzi L2-loptu sadrzi L1-loptu (zbog ||.||inf<=||.||2<=||.||1).
test_pts = unit_ball_boundary(1)  # tacke sa ||x||1 = 1
log_check('Tacke sa ||x||1=1 imaju ||x||2<=1 i ||x||inf<=1 (ugnjezdenje lopti)',
          all(norm_p(pt, 2) <= 1 + 1e-9 and norm_p(pt, np.inf) <= 1 + 1e-9 for pt in test_pts))
print('=> Oblik lopte = kako metrika kaznjava odstupanja (L1 sparse, Linf max-komponenta, L2 izotropno).')

## 4.7 Otvorene lopte i okoline (Def 7.18)

**STA:** definisemo otvorenu loptu `B_r(x) = {y : d(x,y) < r}` i koristimo je da odredimo koji dokumenti su u epsilon-okolini zadatog dokumenta/upita.

**KAKO (Def 7.18):** za centar `x` i radijus `r`, dokument `y` pripada `B_r(x)` ako je `d(x,y) < r`. Crtamo lopte razlicitih radijusa u 2D PCA prostoru i brojimo clanove.

**ZASTO:** ovo je matematicki temelj **radius-based (range) retrieval-a**: umesto "vrati top-k", kazemo "vrati sve unutar rastojanja r". Koristi se u dedup-u, anomaly detection-u i RAG-u sa pragom slicnosti. Primenjujemo ga u sekciji 5.

In [ ]:
def open_ball_members(center, points, r, metric=d_L2):
    # Def 7.18: indeksi tacaka y za koje je d(center, y) < r.
    dists = np.array([metric(center, p) for p in points])
    return np.where(dists < r)[0], dists

# Radimo u 2D PCA prostoru (coords2d iz 3.5) radi vizualizacije.
center_doc = 4  # 'A vector space is closed ...' (linear_algebra)
center = coords2d[center_doc]

log_step('4.7 Otvorene lopte B_r(x) (Def 7.18)', f'centar = dok {center_doc} ("{documents[center_doc][:40]}...")')

fig, ax = plt.subplots(figsize=(8.5, 7))
for lab in sorted(df_docs.label.unique()):
    mask = (y == lab)
    ax.scatter(coords2d[mask, 0], coords2d[mask, 1], label=lab, s=70, alpha=0.8)

dists_all = np.array([d_L2(center, p) for p in coords2d])
for r in [0.15, 0.3, 0.5]:
    members, _ = open_ball_members(center, coords2d, r, metric=d_L2)
    circle = plt.Circle((center[0], center[1]), r, fill=False, ls='--', alpha=0.7)
    ax.add_patch(circle)
    print(f'[BALL] r={r}: {len(members)} clanova -> doc_ids={members.tolist()} '
          f'labele={[labels[i] for i in members]}')

ax.scatter([center[0]], [center[1]], color='black', marker='*', s=250, label=f'centar (dok {center_doc})')
ax.set_aspect('equal')
ax.set_title('Otvorene lopte B_r(x) oko dokumenta u 2D PCA prostoru (Def 7.18)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

log_check('Veci radijus => vise (ili jednako) clanova (monotonija lopte)',
          len(open_ball_members(center, coords2d, 0.5)[0]) >= len(open_ball_members(center, coords2d, 0.15)[0]))
print('=> B_r(x) je osnova range/radius retrieval-a (vrati sve unutar praga r), koristimo u sekciji 5.')

## 4.8 Nizovi, Cauchy niz i kompletnost (Def 7.31, 7.38, 7.39)

**STA:** pokrecemo gradient descent na kvadratnoj funkciji gubitka i pokazujemo da niz iterata `(x_n)` jeste **Cauchy niz** koji **konvergira** ka resenju - upravo onako kako Ch7 definise konvergenciju i kompletnost.

**KAKO:**
- **Def 7.31 (konvergencija):** `x_n → x*` ako `d(x_n, x*) → 0`.
- **Def 7.38 (Cauchy):** za svako `ε>0` postoji `N` tako da `d(x_m, x_n) < ε` za `m,n > N`. Pratimo `d(x_n, x_{n+1})`.
- **Def 7.39 (kompletnost):** u kompletnom prostoru (`R^n` je Banach, Ex 7.41) svaki Cauchy niz konvergira.

**ZASTO:** ovo je *direktna* veza analize i AI treninga. Trening (gradient descent) generise niz parametara u `R^n`; jer je `R^n` kompletan, Cauchy niz garantovano ima granicu *u istom prostoru* - zato trening konvergira ka konkretnim parametrima, a ne "pobegne" van prostora. Crtamo `d(x_n, x*)` i `d(x_n, x_{n+1})` na log skali.

In [ ]:
# Kvadratna funkcija gubitka L(x) = 0.5 (x-x*)^T Q (x-x*); gradient descent treba da konvergira ka x*.
rng = np.random.default_rng(SEED)
dim = 6
Qmat = rng.normal(size=(dim, dim))
Qmat = Qmat.T @ Qmat + np.eye(dim)        # simetricna pozitivno-definitna
x_star = rng.normal(size=dim)             # pravi minimum (limit niza)

def grad(x):
    return Qmat @ (x - x_star)

lr = 1.0 / np.linalg.eigvalsh(Qmat).max()  # stabilan korak
x_curr = rng.normal(size=dim) * 5
iterates = [x_curr.copy()]
for _ in range(60):
    x_curr = x_curr - lr * grad(x_curr)
    iterates.append(x_curr.copy())
iterates = np.array(iterates)

# d(x_n, x*) i d(x_n, x_{n+1}) u L2 metrici.
dist_to_limit = np.array([d_L2(x, x_star) for x in iterates])
step_dist = np.array([d_L2(iterates[i], iterates[i + 1]) for i in range(len(iterates) - 1)])

log_step('4.8 GD iterati kao Cauchy niz', f'dim={dim}, korak lr={lr:.4f}, iteracija={len(iterates) - 1}')
log_check('Konvergencija (Def 7.31): d(x_n, x*) -> 0', dist_to_limit[-1] < 1e-4,
          f'd(x_last, x*)={dist_to_limit[-1]:.3e}')
log_check('Cauchy uslov (Def 7.38): d(x_n, x_{n+1}) -> 0', step_dist[-1] < 1e-4,
          f'd(x_last, x_prev)={step_dist[-1]:.3e}')
log_check('Monotono opadanje rastojanja do limita', np.all(np.diff(dist_to_limit) <= 1e-9))

# Eksplicitan epsilon-N test (Def 7.38): za dato eps nadji N.
eps = 1e-3
N_idx = next((i for i in range(len(step_dist)) if np.all(step_dist[i:] < eps)), None)
print(f'[INFO] Za eps={eps}: svi koraci posle N={N_idx} su < eps (Cauchy uslov ispunjen).')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(dist_to_limit, marker='o', label='d(x_n, x*)  (Def 7.31)')
ax.semilogy(step_dist, marker='s', label='d(x_n, x_{n+1})  (Def 7.38)')
ax.axhline(eps, color='red', ls='--', alpha=0.6, label=f'eps={eps}')
ax.set_title('Gradient descent: konvergentan Cauchy niz u (R^n, L2), Banach (Ex 7.41)')
ax.set_xlabel('iteracija n'); ax.set_ylabel('rastojanje (log skala)'); ax.legend()
plt.tight_layout(); plt.show()
print('=> R^n je kompletan (Banach), pa Cauchy niz iterata GARANTOVANO konvergira u prostoru -> trening "sleti" u tacku.')

# 5. AI primena A: Semantic retrieval po razlicitim metrikama

**STA:** za skup upita (svaki sa ocekivanom temom) poredimo kvalitet retrieval-a koristeci razne mere iz Ch7 (cosine, L2, L1, Linf, Mahalanobis), u punom prostoru (Ch2: ceo `R^d`) i u podprostoru (Ch2: projekcija na `V_k`).

**KAKO:**
- rangiranje: za slicnost (cosine) sortiramo opadajuce, za rastojanja (L1/L2/Linf/Mahalanobis) rastuce,
- metrika kvaliteta: `Recall@k` = udeo upita kod kojih je medju top-k bar jedan dokument tacne teme,
- **Mahalanobis** racunamo u SVD-redukovanom prostoru `Z = X V_k` (tamo je kovarijansa invertibilna) - spaja Ch2 (podprostor) i Ch7 (metrika),
- dodatno radimo **radius (epsilon-ball) retrieval** iz 4.7.

**ZASTO:** ovo je srz semantic search-a. Pokazujemo *empirijski* kako izbor metrike (Ch7) i izbor prostora/baze (Ch2) zajedno odredjuju kvalitet pretrage.

In [ ]:
# Upiti sa ocekivanom temom (label) radi evaluacije.
queries = [
    ('How do attention mechanisms work in transformers?', 'ai'),
    ('What defines a basis and the dimension of a vector space?', 'linear_algebra'),
    ('Explain the triangle inequality and distance between points', 'metric_spaces'),
    ('Tips for balanced nutrition and healthy meals', 'health'),
    ('How to track macroeconomic market volatility', 'finance'),
    ('Strategies for team ball games and scoring', 'sports'),
]
query_texts = [q for q, _ in queries]
query_true = [lab for _, lab in queries]

def encode_queries(texts):
    if USE_SENTENCE_TRANSFORMERS:
        return st_model.encode(texts, convert_to_numpy=True).astype(float)
    return vectorizer.transform(texts).toarray().astype(float)

Q = encode_queries(query_texts)

# Redukovani (subspace) prostor: Z = X V_k, Q_red = Q V_k (Ch2 baza V_k).
Z = X @ V_k
Q_red = Q @ V_k

# Mahalanobis u redukovanom prostoru (kovarijansa je tamo invertibilna).
cov = np.cov(Z, rowvar=False) + 1e-6 * np.eye(Z.shape[1])
VI = np.linalg.inv(cov)
def d_mahalanobis(a, b):
    diff = a - b
    return float(np.sqrt(diff @ VI @ diff))

def rank_docs(q, docs, kind):
    # Vraca indekse dokumenata od najslicnijeg ka najmanje slicnom.
    if kind == 'cosine':
        scores = np.array([cosine_sim(q, d) for d in docs])
        return np.argsort(-scores)
    metric = {'L1': d_L1, 'L2': d_L2, 'Linf': d_Linf, 'mahalanobis': d_mahalanobis}[kind]
    dists = np.array([metric(q, d) for d in docs])
    return np.argsort(dists)

def recall_at_k(Qset, docs, kind, k=3):
    hits = 0
    for i in range(Qset.shape[0]):
        order = rank_docs(Qset[i], docs, kind)[:k]
        if query_true[i] in [labels[j] for j in order]:
            hits += 1
    return hits / Qset.shape[0]

K = 3
configs = [
    ('cosine',      'full',     Q,     X,  'cosine'),
    ('L2',          'full',     Q,     X,  'L2'),
    ('L1',          'full',     Q,     X,  'L1'),
    ('Linf',        'full',     Q,     X,  'Linf'),
    ('cosine',      'subspace', Q_red, Z,  'cosine'),
    ('L2',          'subspace', Q_red, Z,  'L2'),
    ('mahalanobis', 'subspace', Q_red, Z,  'mahalanobis'),
]

log_step('5. Retrieval evaluacija', f'{len(queries)} upita | Recall@{K} | full vs subspace (k={k})')
rows = []
for label_m, space, Qs, Ds, kind in configs:
    r = recall_at_k(Qs, Ds, kind, k=K)
    rows.append({'metric': label_m, 'space': space, f'Recall@{K}': round(r, 3)})
res_df = pd.DataFrame(rows)
display(res_df)

# Detaljan log po upitu za cosine (full).
print('\n--- Detalj po upitu (cosine, full space) ---')
for i, (qt, exp) in enumerate(queries):
    order = rank_docs(Q[i], X, 'cosine')[:K]
    print(f'[Q] "{qt[:55]}..." | ocekivano={exp}')
    print(f'    top-{K} labele = {[labels[j] for j in order]} (doc_ids={order.tolist()})')

# Bar chart Recall po (metric, space).
fig, ax = plt.subplots(figsize=(10, 4.5))
xpos = np.arange(len(res_df))
ax.bar(xpos, res_df[f'Recall@{K}'])
ax.set_xticks(xpos)
ax.set_xticklabels([f"{m}\n({s})" for m, s in zip(res_df['metric'], res_df['space'])])
for i, v in enumerate(res_df[f'Recall@{K}']):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center')
ax.set_ylim(0, 1.1); ax.set_ylabel(f'Recall@{K}')
ax.set_title('Retrieval kvalitet po metrici i prostoru (Ch7 metrike x Ch2 podprostor)')
plt.tight_layout(); plt.show()

In [ ]:
# Radius (epsilon-ball) retrieval (Def 7.18): vrati sve dokumente unutar praga r.
# Koristimo cosine-distance d_cos = 1 - cosine_sim (manje je slicnije).
def cosine_distance(a, b):
    return 1.0 - cosine_sim(a, b)

log_step('5b. Radius retrieval (Def 7.18)', 'B_r(query) u cosine-distance')
radius = 0.6
radius_rows = []
for i, (qt, exp) in enumerate(queries):
    dists = np.array([cosine_distance(Q[i], X[j]) for j in range(X.shape[0])])
    members = np.where(dists < radius)[0]
    member_labels = [labels[j] for j in members]
    precision = (member_labels.count(exp) / len(members)) if len(members) else 0.0
    radius_rows.append({'query': qt[:40] + '...', 'expected': exp,
                        'n_in_ball': len(members), 'precision_in_ball': round(precision, 2)})
    print(f'[BALL r={radius}] "{qt[:45]}..." -> {len(members)} dok, labele={member_labels}')

display(pd.DataFrame(radius_rows))
print('=> Radius retrieval ne fiksira broj rezultata nego prag slicnosti (dedup, RAG sa pragom, anomaly detection).')

# 6. AI primena B: kNN klasifikacija po metrikama

**STA:** treniramo kNN klasifikator tema dokumenata sa razlicitim metrikama (`euclidean`, `manhattan`, `chebyshev`, `cosine`) i poredimo tacnost. Crtamo decision boundary u 2D PCA prostoru.

**KAKO:** kNN za novu tacku gleda `k` najblizih suseda po izabranoj metrici i glasa o klasi. Koristimo `sklearn.KNeighborsClassifier`; trening na train indeksima, evaluacija na test indeksima iz sekcije 2. Granicu odluke vizualizujemo na gustoj mrezi tacaka u 2D.

**ZASTO:** kNN je *najcistiji* primer algoritma koji se u potpunosti oslanja na metricku strukturu (Ch7) nad vektorskim prostorom (Ch2). Nejednakost trougla (4.4) je upravo svojstvo koje omogucava korektno i efikasno trazenje suseda. Razliciti oblici jedinicnih lopti (4.6) daju razlicite granice odluke - sto i vizuelno pokazujemo.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_metrics = ['euclidean', 'manhattan', 'chebyshev', 'cosine']
n_neighbors = 3

X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

log_step('6. kNN klasifikacija', f'k={n_neighbors} | train={len(idx_train)} | test={len(idx_test)}')
acc_rows = []
for met in knn_metrics:
    clf = KNeighborsClassifier(n_neighbors=n_neighbors, metric=met, algorithm='brute')
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    acc_rows.append({'metric': met, 'test_accuracy': round(acc, 3)})
    log_check(f'kNN ({met}) istreniran i evaluiran', True, f'test_accuracy={acc:.3f}')

acc_df = pd.DataFrame(acc_rows)
display(acc_df)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(acc_df['metric'], acc_df['test_accuracy'])
for i, v in enumerate(acc_df['test_accuracy']):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center')
ax.set_ylim(0, 1.1); ax.set_ylabel('test accuracy')
ax.set_title(f'kNN (k={n_neighbors}) tacnost po metrici (Ch7 metrike)')
plt.tight_layout(); plt.show()

In [ ]:
# Decision boundary u 2D PCA prostoru (coords2d iz 3.5) za svaku metriku.
from matplotlib.colors import ListedColormap

classes = sorted(df_docs.label.unique())
class_to_int = {c: i for i, c in enumerate(classes)}
y_int = np.array([class_to_int[c] for c in y])
cmap = ListedColormap(plt.cm.tab10(np.linspace(0, 1, len(classes))))

# Mreza preko 2D prostora.
pad = 0.5
x_min, x_max = coords2d[:, 0].min() - pad, coords2d[:, 0].max() + pad
y_min, y_max = coords2d[:, 1].min() - pad, coords2d[:, 1].max() + pad
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, met in zip(axes.ravel(), knn_metrics):
    clf2d = KNeighborsClassifier(n_neighbors=n_neighbors, metric=met, algorithm='brute')
    clf2d.fit(coords2d, y_int)
    zz = clf2d.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap=cmap, levels=np.arange(-0.5, len(classes) + 0.5))
    for c in classes:
        mask = (y == c)
        ax.scatter(coords2d[mask, 0], coords2d[mask, 1], s=60, label=c,
                   color=cmap(class_to_int[c]), edgecolor='k', linewidth=0.5)
    ax.set_title(f'kNN decision boundary - metrika: {met}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
axes[0, 0].legend(fontsize=7, loc='best')
plt.tight_layout(); plt.show()
print('=> Razlicite metrike (Ch7) -> razliciti oblici granica odluke (uporedi sa jedinicnim loptama iz 4.6).')

# 7. Finalni summary i mapiranje na teoriju

**STA:** sazimamo sve glavne artefakte i rezultate run-a u jednu tabelu i rekapituliramo gde je svaka teorija upotrebljena.

In [ ]:
best_retrieval = res_df.loc[res_df[f'Recall@{K}'].idxmax()]
best_knn = acc_df.loc[acc_df['test_accuracy'].idxmax()]

summary_df = pd.DataFrame([
    {'artifact': 'documents', 'value': len(documents), 'ref': 'dataset'},
    {'artifact': 'X shape', 'value': str(X.shape), 'ref': 'embedding matrica'},
    {'artifact': 'rank(X)', 'value': int(np.linalg.matrix_rank(X)), 'ref': 'Strang 2.3'},
    {'artifact': 'dim N(X)', 'value': N.shape[1], 'ref': 'Strang 2.2/2.4'},
    {'artifact': 'subspace k', 'value': k, 'ref': 'Strang 2.3 (V_k)'},
    {'artifact': 'captured energy', 'value': round(float(captured_energy), 4), 'ref': 'SVD'},
    {'artifact': 'P idempotent & symmetric', 'value': 'PASS', 'ref': 'Strang 2.4'},
    {'artifact': 'sve mere validne metrike', 'value': 'PASS', 'ref': 'Def 7.1'},
    {'artifact': 'Cauchy-Schwarz', 'value': 'PASS', 'ref': 'Thm 7.54'},
    {'artifact': 'Minkowski (trougao)', 'value': 'PASS', 'ref': 'Cor 7.55'},
    {'artifact': 'GD Cauchy konvergencija', 'value': f'd_final={dist_to_limit[-1]:.1e}', 'ref': 'Def 7.38/7.39'},
    {'artifact': f'najbolji retrieval Recall@{K}', 'value': f"{best_retrieval['metric']}/{best_retrieval['space']}={best_retrieval[f'Recall@{K}']}", 'ref': 'Ch2+Ch7'},
    {'artifact': 'najbolji kNN', 'value': f"{best_knn['metric']}={best_knn['test_accuracy']}", 'ref': 'Ch7'},
])

log_step('7. Finalni summary', 'Pregled svih glavnih artefakata i provera.')
display(summary_df)

## Zakljucak: STA / KAKO / ZASTO po sekciji

### Strang, Chapter 2 - Vector Spaces (gde "zive" embeddingi)
- **2.1 (3.1):** embeddingi zadovoljavaju aksiome vektorskog prostora -> smemo da ih sabiramo, skaliramo, projektujemo.
- **2.3 (3.2):** RREF daje eksplicitnu bazu i `rank` = stvarna (efektivna) dimenzionalnost reprezentacije.
- **2.2 (3.3):** `Ax=b` (partikularno) + `N(X)` (homogeno) = sva resenja; nullspace = "slepi" pravci / redundansa.
- **2.4 (3.4):** cetiri fundamentalna podprostora + rank-nullity + ortogonalnost objasnjavaju gde je signal a gde redundansa.
- **2.1/2.3/2.4 (3.5):** SVD baza `V_k` i ortogonalna projekcija `P=V_k V_k^T` (`P^2=P`, `P=P^T`) = PCA/LSA; residual = mera novine.

### Introduction to Analysis, Chapter 7 - Metric Spaces (kako merimo i zasto trening konvergira)
- **Def 7.1 (4.1):** sve mere koje koristimo su validne metrike -> retrieval/kNN su matematicki ispravni.
- **Def 7.11 / Prop 7.12 (4.2):** metrika iz norme; translaciona invarijantnost i homogenost opravdavaju centriranje/skaliranje.
- **Thm 7.54 (4.3):** Cauchy-Schwarz cini cosine slicnost ogranicenom na `[-1,1]` i interpretabilnom.
- **Cor 7.55 (4.4):** Minkowski / nejednakost trougla -> korektnost i pruning u kNN-u.
- **str. 98 (4.5):** ekvivalencija normi -> izbor norme menja skalu, ne topologiju.
- **Fig 1 (4.6):** oblik jedinicne lopte objasnjava ponasanje L1/L2/Linf (videti i kNN granice).
- **Def 7.18 (4.7):** otvorene lopte = osnova radius/range retrieval-a.
- **Def 7.31/7.38/7.39 (4.8):** GD iterati su Cauchy niz; kompletnost `R^n` (Ex 7.41) garantuje konvergenciju treninga.

### Sinteza (5, 6)
Embeddingi zive u **vektorskom prostoru (Ch2)**, biramo koristan **podprostor (V_k)**, a slicnost/rastojanje merimo **metrikom (Ch7)**. Retrieval i kNN su mesto gde se obe teorije sastaju: izbor prostora i izbor metrike zajedno odredjuju kvalitet AI sistema.